# Stage 8.2 -- Map-Ready Feature Refactor

Inspect registry-backed feature generation and Mirage compatibility audits.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

AUDIT = Path('../data/gold/feature_audit')
MAPS = Path('../data/gold/maps/map_registry')
CONTRACT = Path('../data/gold/features/feature_contract')

def load_table(base, name):
    return pd.read_parquet(base / f'{name}.parquet')

registry = load_table(MAPS, 'map_registry')
regions = load_table(MAPS, 'map_region_registry')
contract = load_table(CONTRACT, 'feature_contract')
usage = load_table(AUDIT, 'map_feature_registry_usage')
compatibility = load_table(AUDIT, 'map_feature_compatibility')
unknowns = load_table(AUDIT, 'map_feature_unknowns')
refactor_audit = load_table(AUDIT, 'map_feature_refactor_audit')

## Registry

In [ ]:
display(registry)
display(regions[['region_id', 'display_name', 'geometry_type', 'geometry_source', 'semantic_tags', 'aliases']])

## Feature Contract Scopes

In [ ]:
scope_counts = contract['map_scope'].value_counts(dropna=False).rename_axis('map_scope').reset_index(name='features')
display(scope_counts)
ax = scope_counts.plot.bar(x='map_scope', y='features', legend=False, figsize=(6, 3))
ax.set_xlabel('')
ax.set_ylabel('features')
plt.tight_layout()

## Registry Usage

In [ ]:
display(usage.groupby(['map_scope', 'resolution_source', 'resolution_status']).size().reset_index(name='features'))
display(usage[usage['region_dependency']].head(30))

## Compatibility

In [ ]:
display(compatibility.groupby(['dataset_name', 'status']).size().reset_index(name='checks'))
display(compatibility[compatibility['status'] != 'ok'].head(30))

## Candidate Feature Compatibility

In [ ]:
candidate_checks = compatibility[compatibility['is_candidate_feature']]
display(candidate_checks.groupby('status').size().reset_index(name='candidate_checks'))
display(candidate_checks[candidate_checks['status'] != 'ok'])

## Unknowns And Final Audit

In [ ]:
display(unknowns)
display(refactor_audit)

Next: run Mirage backward-compatibility gate before onboarding a new map.